In [2]:
# ===============================================================================
# INTERACTIVE FUNCTIONS
# ===============================================================================

def run_interactive():
    """
    Interactive function to run fall detection with user input
    """
    # Check if required modules are imported
    try:
        import os
        import time
        import re
    except ImportError as e:
        print(f"Error: Required module not available: {e}")
        return
    
    # Get the weights file
    poseweights = input("Enter path to weights file [default: yolov7-w6-pose.pt]: ") or "yolov7-w6-pose.pt"
    
    # Get device type
    use_gpu = input("Use GPU? (y/n) [default: y]: ").lower() or "y"
    if use_gpu == "y":
        device_input = input("Enter GPU device ID [default: 0]: ").strip()
        # Fix: Handle case where user enters "y" or other invalid input
        if device_input == "" or device_input.lower() == "y":
            device = "0"
        elif device_input.isdigit():
            device = device_input
        else:
            print(f"Invalid device ID '{device_input}'. Using default: 0")
            device = "0"
    else:
        device = "cpu"
    
    # Get source type
    print("\nSelect input source:")
    print("1: Video file")
    print("2: Webcam")
    print("3: Single video from Le2i dataset")
    print("4: Interactive single video processing from Le2i dataset")
    print("5: Process all Le2i dataset videos")
    source_choice = input("Enter choice [1/2/3/4/5]: ")
    
    if source_choice == "1":
        # Video file
        default_video = "sample_video.mp4"
        source = input(f"Enter video file path [default: {default_video}]: ") or default_video
        # Ask if user wants to display the processed video in real-time
        display_video = input("Display video with pose estimation in real-time? (y/n) [default: y]: ").lower() or "y"
        # Ask if user wants to save the output video
        save_video = input("Save output video? (y/n) [default: y]: ").lower() or "y"
        
        print(f"\nRunning fall detection with:")
        print(f"- Weights: {poseweights}")
        print(f"- Device: {device}")
        print(f"- Source: {source}")
        print(f"- Display: {'Yes' if display_video == 'y' else 'No'}")
        print(f"- Save output: {'Yes' if save_video == 'y' else 'No'}")
        confirmation = input("\nConfirm? (y/n) [default: y]: ").lower() or "y"
        
        if confirmation == "y":
            # Run the model
            run_with_display = (display_video == "y")
            save_output = (save_video == "y")
            
            # Try to strip optimizer to ensure model works correctly (optional)
            try:
                strip_optimizer(device, poseweights)
            except (NameError, ImportError, AttributeError) as e:
                print(f"Note: Could not strip optimizer ({e}). Continuing anyway...")
            except Exception as e:
                print(f"Warning: Error during optimizer stripping ({e}). Continuing anyway...")
            
            # Run fall detection
            run_fall_detection(
                poseweights=poseweights,
                source=source,
                device=device,
                display=run_with_display,
                save_output=save_output
            )
        else:
            print("Operation cancelled")
    
    elif source_choice == "2":
        # Webcam
        cam_id = input("Enter webcam ID [default: 0]: ") or "0"
        source = cam_id
        
        # Ask if user wants to save the output video
        save_video = input("Save output video? (y/n) [default: y]: ").lower() or "y"
        
        print(f"\nRunning fall detection with:")
        print(f"- Weights: {poseweights}")
        print(f"- Device: {device}")
        print(f"- Source: Webcam {source}")
        print(f"- Display: Yes")  # Always display for webcam
        print(f"- Save output: {'Yes' if save_video == 'y' else 'No'}")
        confirmation = input("\nConfirm? (y/n) [default: y]: ").lower() or "y"
        
        if confirmation == "y":
            # Try to strip optimizer to ensure model works correctly (optional)
            try:
                strip_optimizer(device, poseweights)
            except (NameError, ImportError, AttributeError) as e:
                print(f"Note: Could not strip optimizer ({e}). Continuing anyway...")
            except Exception as e:
                print(f"Warning: Error during optimizer stripping ({e}). Continuing anyway...")
            
            # Run fall detection
            run_fall_detection(
                poseweights=poseweights,
                source=source,
                device=device,
                display=True,  # Always display for webcam
                save_output=(save_video == "y")
            )
        else:
            print("Operation cancelled")
    
    elif source_choice == "3":
        # Single video from Le2i dataset
        default_dataset_path = "datasets"
        dataset_path = input(f"Enter dataset root path [default: {default_dataset_path}]: ") or default_dataset_path
        
        # Check if the dataset path exists
        if not os.path.exists(dataset_path):
            print(f"Error: Dataset path '{dataset_path}' does not exist.")
            return
        
        # Check for Le2i dataset structure
        le2i_path = os.path.join(dataset_path, "le2i")
        if not os.path.exists(le2i_path):
            print(f"Error: Le2i dataset not found at {le2i_path}")
            return
        
        # Check for Le2i_Sorted structure
        le2i_sorted_path = os.path.join(le2i_path, "Le2i_Sorted")
        is_sorted = os.path.exists(le2i_sorted_path)
        
        if is_sorted:
            print(f"Found Le2i dataset with sorted structure (Fall/Non Fall folders)")
            # Get environment folder
            env_folders = ["Coffee_room_01", "Coffee_room_02", "Home_01", "Home_02", "Lecture_room", "Office"]
            print("Choose environment folder:")
            for i, env in enumerate(env_folders, 1):
                print(f"{i}: {env}")
            env_choice = input("Enter choice [1-6, default: 1]: ") or "1"
            try:
                env_index = int(env_choice) - 1
                if 0 <= env_index < len(env_folders):
                    env_folder = env_folders[env_index]
                else:
                    print("Invalid choice. Using Coffee_room_01.")
                    env_folder = env_folders[0]
            except ValueError:
                print("Invalid input. Using Coffee_room_01.")
                env_folder = env_folders[0]
                
            # Choose Fall or Non Fall
            fall_type = input("Choose 'Fall' or 'Non Fall' [default: Fall]: ").strip() or "Fall"
            if fall_type.lower() not in ["fall", "non fall"]:
                print("Invalid choice. Using 'Fall'.")
                fall_type = "Fall"
                
            # Construct path to Videos folder
            videos_folder = os.path.join(le2i_path, "Le2i_Sorted", fall_type, env_folder, "Videos")
            if not os.path.exists(videos_folder):
                print(f"Error: Videos folder not found at {videos_folder}")
                return
                
            # List available videos with natural sorting
            video_files = [f for f in os.listdir(videos_folder) 
                         if f.endswith(('.mp4', '.avi', '.mov', '.mkv', '.MP4', '.AVI', '.MOV', '.MKV'))]
            
            # FIXED: Sort videos naturally so they appear in correct numerical order
            video_files.sort(key=natural_sort_key)
            
            if not video_files:
                print(f"Error: No video files found in {videos_folder}")
                return
                
            print(f"Found {len(video_files)} video files.")
            print("Choose video file:")
            for i, video in enumerate(video_files, 1):
                print(f"{i}: {video}")
                
            video_choice = input(f"Enter choice [1-{len(video_files)}, default: 1]: ") or "1"
            try:
                video_index = int(video_choice) - 1
                if 0 <= video_index < len(video_files):
                    video_file = video_files[video_index]
                else:
                    print(f"Invalid choice. Using {video_files[0]}.")
                    video_file = video_files[0]
            except ValueError:
                print(f"Invalid input. Using {video_files[0]}.")
                video_file = video_files[0]
                
            # Construct full path to video file
            source = os.path.join(videos_folder, video_file)
        else:
            print("Le2i dataset with traditional structure not supported for this option.")
            return
            
        # Ask if user wants to display the processed video in real-time
        display_video = input("Display video with pose estimation in real-time? (y/n) [default: y]: ").lower() or "y"
        
        # Ask if user wants to save the output video
        save_video = input("Save output video? (y/n) [default: y]: ").lower() or "y"
        
        # Ask if user wants to save detections to a file
        save_detections = input("Save frame-by-frame detections to a file? (y/n) [default: y]: ").lower() or "y"
        
        print(f"\nRunning fall detection with:")
        print(f"- Weights: {poseweights}")
        print(f"- Device: {device}")
        print(f"- Source: {source}")
        print(f"- Display: {'Yes' if display_video == 'y' else 'No'}")
        print(f"- Save output: {'Yes' if save_video == 'y' else 'No'}")
        print(f"- Save detections: {'Yes' if save_detections == 'y' else 'No'}")
        print(f"- Fall video comparison: Enabled (will check annotations)")
        confirmation = input("\nConfirm? (y/n) [default: y]: ").lower() or "y"
        
        if confirmation == "y":
            # Try to strip optimizer to ensure model works correctly (optional)
            try:
                strip_optimizer(device, poseweights)
            except (NameError, ImportError, AttributeError) as e:
                print(f"Note: Could not strip optimizer ({e}). Continuing anyway...")
            except Exception as e:
                print(f"Warning: Error during optimizer stripping ({e}). Continuing anyway...")
            
            # Run fall detection using the same function as the other options
            results = run_fall_detection(
                poseweights=poseweights,
                source=source,
                device=device,
                display=(display_video == "y"),
                save_output=(save_video == "y"),
                save_detections=(save_detections == "y")
            )
            
            # Display summarized results
            if results:
                print("\nProcessing Summary:")
                print(f"Processed {results['frames_processed']} frames")
                print(f"Average FPS: {results['average_fps']:.2f}")
                print(f"False detections: {results['false_detections']}")
                
                if results['false_detections'] > 0 and save_detections == 'y':
                    print(f"Detection details saved to: {results['detections_path']}")
        else:
            print("Operation cancelled")
    
    elif source_choice == "4":
        # NEW OPTION: Interactive single video processing from Le2i dataset
        default_dataset_path = "datasets"
        dataset_path = input(f"Enter dataset root path [default: {default_dataset_path}]: ") or default_dataset_path
        
        # Check if the dataset path exists
        if not os.path.exists(dataset_path):
            print(f"Error: Dataset path '{dataset_path}' does not exist.")
            return
        
        # Check for Le2i dataset structure
        le2i_path = os.path.join(dataset_path, "le2i")
        if not os.path.exists(le2i_path):
            print(f"Error: Le2i dataset not found at {le2i_path}")
            return
        
        # Check for Le2i_Sorted structure
        le2i_sorted_path = os.path.join(le2i_path, "Le2i_Sorted")
        is_sorted = os.path.exists(le2i_sorted_path)
        
        if not is_sorted:
            print("Error: Le2i dataset with sorted structure (Fall/Non Fall folders) not found.")
            return
            
        print(f"Found Le2i dataset with sorted structure at {le2i_sorted_path}")
        
        # Get user preferences once
        display_videos = input("Display videos with pose estimation in real-time? (y/n) [default: y]: ").lower() or "y"
        save_videos = input("Save output videos? (y/n) [default: y]: ").lower() or "y"
        save_detections = input("Save frame-by-frame detections to a file? (y/n) [default: y]: ").lower() or "y"
        
        # Try to strip optimizer to ensure model works correctly (optional)
        try:
            strip_optimizer(device, poseweights)
        except (NameError, ImportError, AttributeError) as e:
            print(f"Note: Could not strip optimizer ({e}). Continuing anyway...")
        except Exception as e:
            print(f"Warning: Error during optimizer stripping ({e}). Continuing anyway...")
        
        # Initialize FallDetector once for all videos
        detector = FallDetector(poseweights=poseweights, device=device)
        
        # Environment folders
        env_folders = ["Coffee_room_01", "Coffee_room_02", "Home_01", "Home_02", "Lecture_room", "Office"]
        fall_folders = ['Fall', 'Non Fall']
        
        # REVISED: Track cumulative metrics across all videos in the session
        session_videos_processed = 0
        session_true_positives = 0
        session_true_negatives = 0
        session_false_positives = 0
        session_false_negatives = 0
        session_total_frames = 0
        session_total_detected_fall_frames = 0
        session_total_detected_non_fall_frames = 0
        
        # Interactive loop
        while True:
            print("\n" + "="*50)
            print("INTERACTIVE VIDEO PROCESSING")
            print("="*50)
            
            # Choose environment folder
            print("Choose environment folder:")
            for i, env in enumerate(env_folders, 1):
                print(f"{i}: {env}")
            print("0: Exit")
            
            env_choice = input("Enter choice [0-6]: ")
            if env_choice == "0":
                print("Exiting interactive mode.")
                break
            
            try:
                env_index = int(env_choice) - 1
                if 0 <= env_index < len(env_folders):
                    env_folder = env_folders[env_index]
                else:
                    print("Invalid choice. Please try again.")
                    continue
            except ValueError:
                print("Invalid input. Please try again.")
                continue
            
            # Choose Fall or Non Fall
            print("\nChoose fall type:")
            print("1: Fall")
            print("2: Non Fall")
            print("0: Back to environment selection")
            
            fall_choice = input("Enter choice [0-2]: ")
            if fall_choice == "0":
                continue
            elif fall_choice == "1":
                fall_type = "Fall"
            elif fall_choice == "2":
                fall_type = "Non Fall"
            else:
                print("Invalid choice. Please try again.")
                continue
            
            # Construct path to Videos folder
            videos_folder = os.path.join(le2i_path, "Le2i_Sorted", fall_type, env_folder, "Videos")
            if not os.path.exists(videos_folder):
                print(f"Error: Videos folder not found at {videos_folder}")
                continue
                
            # List available videos with natural sorting
            video_files = [f for f in os.listdir(videos_folder) 
                         if f.endswith(('.mp4', '.avi', '.mov', '.mkv', '.MP4', '.AVI', '.MOV', '.MKV'))]
            
            # FIXED: Sort videos naturally so they appear in correct numerical order
            video_files.sort(key=natural_sort_key)
            
            if not video_files:
                print(f"Error: No video files found in {videos_folder}")
                continue
                
            print(f"\nFound {len(video_files)} video files in {env_folder}/{fall_type}:")
            for i, video in enumerate(video_files, 1):
                print(f"{i}: {video}")
            print("0: Back to fall type selection")
                
            video_choice = input(f"Enter choice [0-{len(video_files)}]: ")
            if video_choice == "0":
                continue
            
            try:
                video_index = int(video_choice) - 1
                if 0 <= video_index < len(video_files):
                    video_file = video_files[video_index]
                else:
                    print("Invalid choice. Please try again.")
                    continue
            except ValueError:
                print("Invalid input. Please try again.")
                continue
                
            # Construct full path to video file
            source = os.path.join(videos_folder, video_file)
            
            print(f"\nProcessing: {env_folder}/{fall_type}/{video_file}")
            print(f"Settings: Display={display_videos=='y'}, Save={save_videos=='y'}, Save Detections={save_detections=='y'}")
            print(f"Video path: {source}")
            
            # Check if file exists and is readable
            if not os.path.exists(source):
                print(f"❌ ERROR: Video file does not exist: {source}")
                continue
            
            # Check file size
            file_size = os.path.getsize(source)
            if file_size == 0:
                print(f"❌ ERROR: Video file is empty (0 bytes): {source}")
                continue
            
            print(f"File size: {file_size/1024/1024:.2f} MB")
            
            # Process the selected video
            try:
                results = run_fall_detection(
                    detector=detector,  # Reuse the same detector
                    source=source,
                    device=device,
                    display=(display_videos == "y"),
                    save_output=(save_videos == "y"),
                    save_detections=(save_detections == "y"),
                    batch_mode=False,  # Show individual results
                    interactive_mode=True  # Enable interactive mode to suppress individual metrics
                )
                
                # Check if video processing was successful
                if results and results['frames_processed'] > 0:
                    # REVISED: Accumulate session metrics only for successful videos
                    session_videos_processed += 1
                    session_true_positives += results['true_positives']
                    session_true_negatives += results['true_negatives']
                    session_false_positives += results['false_positives']
                    session_false_negatives += results['false_negatives']
                    session_total_frames += results['frames_processed']
                    session_total_detected_fall_frames += results['detected_fall_frames']
                    session_total_detected_non_fall_frames += results['detected_non_fall_frames']
                    
                    # Display individual video results (simplified)
                    print(f"\nCurrent Video Results:")
                    print(f"Processed {results['frames_processed']} frames")
                    print(f"Average FPS: {results['average_fps']:.2f}")
                    print(f"Detected fall frames: {results['detected_fall_frames']}")
                    print(f"Detected non-fall frames: {results['detected_non_fall_frames']}")
                    
                    if results['false_detections'] > 0 and save_detections == 'y':
                        print(f"Detection details saved to: {results['detections_path']}")
                        
                elif results and 'error' in results:
                    # Video processing failed
                    print(f"\n❌ ERROR: Failed to process video - {results['error']}")
                    print("This video will be skipped and not counted in session metrics.")
                    continue  # Skip to next iteration without counting this video
                else:
                    # Unknown failure
                    print(f"\n❌ ERROR: Unknown failure processing video {video_file}")
                    print("This video will be skipped and not counted in session metrics.")
                    continue  # Skip to next iteration without counting this video
                
                # REVISED: Show cumulative session metrics only after successful processing
                print(f"\nSession Cumulative Metrics:")
                print(f"True Positives: {session_true_positives}")
                print(f"True Negatives: {session_true_negatives}")
                print(f"False Positives: {session_false_positives}")
                print(f"False Negatives: {session_false_negatives}")
                print(f"Total videos processed: {session_videos_processed}/{session_videos_processed}")
                
                # Calculate session performance metrics
                if session_videos_processed > 0:
                    session_accuracy = (session_true_positives + session_true_negatives) / session_videos_processed
                    session_precision = session_true_positives / max(1, session_true_positives + session_false_positives) if (session_true_positives + session_false_positives) > 0 else 0
                    session_recall = session_true_positives / max(1, session_true_positives + session_false_negatives) if (session_true_positives + session_false_negatives) > 0 else 0
                    session_f1 = 2 * session_precision * session_recall / max(0.001, session_precision + session_recall) if (session_precision + session_recall) > 0 else 0
                    
                    print(f"Session Accuracy: {session_accuracy*100:.2f}%")
                    print(f"Session Precision: {session_precision*100:.2f}%")
                    print(f"Session Recall: {session_recall*100:.2f}%")
                    print(f"Session F1 Score: {session_f1*100:.2f}%")
                
            except Exception as e:
                print(f"❌ CRITICAL ERROR processing video {video_file}: {str(e)}")
                print("This video will be skipped and not counted in session metrics.")
                import traceback
                traceback.print_exc()
                continue  # Skip to next iteration without counting this video
            
            # Ask if user wants to continue
            continue_choice = input("\nProcess another video? (y/n) [default: y]: ").lower() or "y"
            if continue_choice != "y":
                break
    
    elif source_choice == "5":
        # Process all Le2i dataset videos (moved from option 4)
        default_dataset_path = "datasets"
        dataset_path = input(f"Enter dataset root path [default: {default_dataset_path}]: ") or default_dataset_path
        
        # Check if the dataset path exists
        if not os.path.exists(dataset_path):
            print(f"Error: Dataset path '{dataset_path}' does not exist.")
            return
        
        # Check for Le2i dataset structure
        le2i_path = os.path.join(dataset_path, "le2i")
        if not os.path.exists(le2i_path):
            print(f"Error: Le2i dataset not found at {le2i_path}")
            return
        
        # Check for Le2i_Sorted structure
        le2i_sorted_path = os.path.join(le2i_path, "Le2i_Sorted")
        is_sorted = os.path.exists(le2i_sorted_path)
        
        if not is_sorted:
            print("Error: Le2i dataset with sorted structure (Fall/Non Fall folders) not found.")
            return
            
        print(f"Found Le2i dataset with sorted structure at {le2i_sorted_path}")
        
        # Ask if user wants to display the processed videos in real-time
        display_videos = input("Display videos with pose estimation in real-time? (y/n) [default: n]: ").lower() or "n"
        
        # Ask if user wants to save the output videos
        save_videos = input("Save output videos? (y/n) [default: y]: ").lower() or "y"
        
        # Ask if user wants to save detections to a file
        save_detections = input("Save frame-by-frame detections to a file? (y/n) [default: y]: ").lower() or "y"
        
        # Create summary stats file
        timestamp = time.strftime("%Y%m%d-%H%M%S")
        summary_path = os.path.join('output', 'le2i_results', f'batch_summary_{timestamp}.txt')
        os.makedirs(os.path.dirname(summary_path), exist_ok=True)
        
        print(f"\nRunning batch processing of all Le2i dataset videos with:")
        print(f"- Weights: {poseweights}")
        print(f"- Device: {device}")
        print(f"- Display: {'Yes' if display_videos == 'y' else 'No'}")
        print(f"- Save outputs: {'Yes' if save_videos == 'y' else 'No'}")
        print(f"- Save detections: {'Yes' if save_detections == 'y' else 'No'}")
        print(f"- Summary will be saved to: {summary_path}")
        confirmation = input("\nConfirm? (y/n) [default: y]: ").lower() or "y"
        
        if confirmation == "y":
            # Try to strip optimizer to ensure model works correctly (optional)
            try:
                strip_optimizer(device, poseweights)
            except (NameError, ImportError, AttributeError) as e:
                print(f"Note: Could not strip optimizer ({e}). Continuing anyway...")
            except Exception as e:
                print(f"Warning: Error during optimizer stripping ({e}). Continuing anyway...")
            
            # Initialize FallDetector once for all videos
            detector = FallDetector(poseweights=poseweights, device=device)
            
            # Get list of all environment folders
            fall_folders = ['Fall', 'Non Fall']
            env_folders = ["Coffee_room_01", "Coffee_room_02", "Home_01", "Home_02", "Lecture_room", "Office"]

            # Track false detections for summary
            false_detections_data = []
            
            # Initialize counters for summary - REVISED: Count videos and calculate metrics from video-level results
            total_videos = 0
            total_videos_processed = 0
            total_frames = 0
            total_detected_fall_frames = 0
            total_detected_non_fall_frames = 0
            total_processing_time = 0
            
            # Video-level metrics tracking
            batch_true_positives = 0
            batch_true_negatives = 0 
            batch_false_positives = 0
            batch_false_negatives = 0
            
            # Open summary file
            with open(summary_path, 'w') as summary_file:
                summary_file.write(f"Le2i Dataset Batch Processing Summary - {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
                summary_file.write("=" * 80 + "\n\n")
                
                # Process all fall types
                for fall_type in fall_folders:
                    summary_file.write(f"\n{fall_type} Videos:\n")
                    summary_file.write("-" * 50 + "\n")
                    
                    # Process all environment folders
                    for env_folder in env_folders:
                        videos_folder = os.path.join(le2i_path, "Le2i_Sorted", fall_type, env_folder, "Videos")
                        
                        # Skip if folder doesn't exist
                        if not os.path.exists(videos_folder):
                            continue
                        
                        # Get video files with natural sorting
                        video_files = [f for f in os.listdir(videos_folder) 
                                     if f.endswith(('.mp4', '.avi', '.mov', '.mkv', '.MP4', '.AVI', '.MOV', '.MKV'))]
                        
                        # FIXED: Sort videos naturally so they are processed in correct numerical order
                        video_files.sort(key=natural_sort_key)
                        
                        if not video_files:
                            continue
                        
                        total_videos += len(video_files)
                        summary_file.write(f"\nEnvironment: {env_folder} - {len(video_files)} videos\n")
                        summary_file.flush()  # Flush to ensure progress is saved
                        
                        # Process each video with unlimited retry logic
                        for video_file in video_files:
                            source = os.path.join(videos_folder, video_file)
                            print(f"\nProcessing: {fall_type}/{env_folder}/{video_file}")
                            
                            # Unlimited retry logic for failed videos
                            expected_fall_type = fall_type
                            retry_count = 0
                            processed_successfully = False
                            
                            while not processed_successfully:
                                if retry_count > 0:
                                    print(f"Retry attempt {retry_count} for {video_file}")
                                    # Wait a bit before retrying, with increasing delay
                                    wait_time = min(5, 2 + retry_count * 0.5)  # Cap at 5 seconds
                                    print(f"Waiting {wait_time:.1f} seconds before retry...")
                                    time.sleep(wait_time)
                                
                                start_time = time.time()
                                
                                # Run fall detection
                                try:
                                    results = run_fall_detection(
                                        detector=detector,  # Reuse the same detector
                                        source=source,
                                        device=device,
                                        display=(display_videos == "y"),
                                        save_output=(save_videos == "y"),
                                        save_detections=(save_detections == "y"),
                                        batch_mode=True  # Enable batch mode to suppress individual metrics
                                    )
                                    
                                    end_time = time.time()
                                    processing_time = end_time - start_time
                                    
                                    # Check if processing was actually completed
                                    if results and results['frames_processed'] > 0:
                                        processed_successfully = True
                                        total_videos_processed += 1
                                        total_frames += results['frames_processed']
                                        total_processing_time += processing_time
                                        
                                        # REVISED: Accumulate frame counts and video-level metrics
                                        total_detected_fall_frames += results['detected_fall_frames']
                                        total_detected_non_fall_frames += results['detected_non_fall_frames']
                                        
                                        # Accumulate video-level metrics
                                        batch_true_positives += results['true_positives']
                                        batch_true_negatives += results['true_negatives']
                                        batch_false_positives += results['false_positives']
                                        batch_false_negatives += results['false_negatives']

                                        if results['false_positives'] > 0:
                                            false_detections_data.append({
                                                'type': 'FP',
                                                'video_path': source,
                                                'false_frames': results['detected_fall_frames'],
                                                'total_false_frames': results['detected_fall_frames'],
                                                'video_name': video_file,
                                                'environment': env_folder,
                                                'expected_type': expected_fall_type
                                            })
                                        elif results['false_negatives'] > 0:
                                            # For Le2i videos, we know the ground truth fall frames from annotations
                                            # But since we're outside run_fall_detection, we can't access that info
                                            # So we'll use the video type to determine expected behavior
                                            if expected_fall_type == "Fall":
                                                # For Fall videos that weren't detected, we don't know exact frame count
                                                missed_frames_info = "Fall video not detected"
                                            else:
                                                # For Non Fall videos, there shouldn't be any fall frames
                                                missed_frames_info = "N/A (Non Fall video)"
                                            
                                            false_detections_data.append({
                                                'type': 'FN',
                                                'video_path': source,
                                                'missed_frames': missed_frames_info,
                                                'total_missed_frames': missed_frames_info,
                                                'video_name': video_file,
                                                'environment': env_folder,
                                                'expected_type': expected_fall_type
                                            })
                                        
                                        # Display individual video results (like interactive mode)
                                        print(f"\nCurrent Video Results:")
                                        print(f"Processed {results['frames_processed']} frames")
                                        print(f"Average FPS: {results['average_fps']:.2f}")
                                        print(f"Detected fall frames: {results['detected_fall_frames']}")
                                        print(f"Detected non-fall frames: {results['detected_non_fall_frames']}")
                                        
                                        if results['detections_path']:
                                            print(f"Detection details saved to: {results['detections_path']}")
                                        
                                        # Show individual video metrics with descriptive note
                                        print(f"\nIndividual Video Metrics:")
                                        print(f"True Positives: {results['true_positives']}")
                                        print(f"True Negatives: {results['true_negatives']}")
                                        print(f"False Positives: {results['false_positives']}")
                                        print(f"False Negatives: {results['false_negatives']}")
                                        
                                        # Add descriptive note
                                        metric_note = get_video_metric_note(
                                            results['true_positives'],
                                            results['true_negatives'],
                                            results['false_positives'],
                                            results['false_negatives'],
                                            expected_fall_type == "Fall",
                                            results['detected_fall_frames'],  # Add detected fall frames
                                            detector.MINIMUM_FALL_FRAMES  # Add minimum threshold
                                        )
                                        print(f"Note: {metric_note}")
                                        
                                        # if results['accuracy'] >= 0:
                                        #     print(f"Video Accuracy: {results['accuracy']*100:.2f}%")
                                        # if results['precision'] >= 0:
                                        #     print(f"Video Precision: {results['precision']*100:.2f}%")
                                        # if results['recall'] >= 0:
                                        #     print(f"Video Recall: {results['recall']*100:.2f}%")
                                        # if results['f1_score'] >= 0:
                                        #     print(f"Video F1 Score: {results['f1_score']*100:.2f}%")
                                        
                                        # Show cumulative batch metrics
                                        print(f"\nBatch Cumulative Metrics:")
                                        print(f"True Positives: {batch_true_positives}")
                                        print(f"True Negatives: {batch_true_negatives}")
                                        print(f"False Positives: {batch_false_positives}")
                                        print(f"False Negatives: {batch_false_negatives}")
                                        print(f"Total videos processed: {total_videos_processed}/{total_videos}")
                                        
                                        # Calculate and show cumulative performance metrics
                                        if total_videos_processed > 0:
                                            batch_accuracy = (batch_true_positives + batch_true_negatives) / total_videos_processed
                                            batch_precision = batch_true_positives / max(1, batch_true_positives + batch_false_positives) if (batch_true_positives + batch_false_positives) > 0 else 0
                                            batch_recall = batch_true_positives / max(1, batch_true_positives + batch_false_negatives) if (batch_true_positives + batch_false_negatives) > 0 else 0
                                            batch_f1 = 2 * batch_precision * batch_recall / max(0.001, batch_precision + batch_recall) if (batch_precision + batch_recall) > 0 else 0
                                            
                                            print(f"Batch Accuracy: {batch_accuracy*100:.2f}%")
                                            print(f"Batch Precision: {batch_precision*100:.2f}%")
                                            print(f"Batch Recall: {batch_recall*100:.2f}%")
                                            print(f"Batch F1 Score: {batch_f1*100:.2f}%")
                                        
                                        print(f"" + "="*60)  # Separator for next video
                                        
                                        # Write results to summary
                                        summary_file.write(f"  - {video_file}: {results['frames_processed']} frames, ")
                                        summary_file.write(f"FPS: {results['average_fps']:.2f}")
                                        summary_file.write(f", Detected Fall Frames: {results['detected_fall_frames']}")
                                        summary_file.write(f", Detected Non-Fall Frames: {results['detected_non_fall_frames']}")
                                        summary_file.write(f", TP: {results['true_positives']}, TN: {results['true_negatives']}")
                                        summary_file.write(f", FP: {results['false_positives']}, FN: {results['false_negatives']}")
                                        
                                        if results['accuracy'] >= 0:
                                            summary_file.write(f", Accuracy: {results['accuracy']*100:.2f}%")
                                        
                                        if retry_count > 0:
                                            summary_file.write(f" (succeeded after {retry_count} retries)")
                                        
                                        summary_file.write("\n")
                                        summary_file.flush()  # Flush to ensure progress is saved
                                        print(f"✅ Successfully processed {video_file}" + (f" (after {retry_count} retries)" if retry_count > 0 else ""))
                                        
                                    elif results and 'error' in results:
                                        print(f"❌ Error: {results['error']}")
                                        retry_count += 1
                                        if retry_count % 10 == 0:  # Every 10 retries, show progress
                                            print(f"⚠️  Still trying {video_file} - attempt {retry_count}")
                                    else:
                                        print(f"❌ No frames processed (unknown error)")
                                        retry_count += 1
                                        if retry_count % 10 == 0:  # Every 10 retries, show progress
                                            print(f"⚠️  Still trying {video_file} - attempt {retry_count}")
                                        
                                except KeyboardInterrupt:
                                    print(f"\n🛑 User interrupted processing. Stopping batch process.")
                                    print(f"Last video being processed: {video_file}")
                                    print(f"You can resume processing from this point later.")
                                    return  # Exit the entire function
                                    
                                except Exception as e:
                                    print(f"❌ Exception processing {video_file}: {str(e)}")
                                    retry_count += 1
                                    if retry_count % 10 == 0:  # Every 10 retries, show progress
                                        print(f"⚠️  Still trying {video_file} - attempt {retry_count}")
                                        print(f"    Last error: {str(e)}")
                            
                            # This line should never be reached since we loop until success
                            # But keeping it for safety
                            if not processed_successfully:
                                print(f"❌ CRITICAL: Somehow exited retry loop without processing {video_file}")
                                summary_file.write(f"  - {video_file}: CRITICAL ERROR - exited retry loop\n")
                                summary_file.flush()
                
                # REVISED: Calculate overall metrics from video-level results
                if total_videos_processed > 0:
                    # Calculate metrics based on video-level results (not frame-level)
                    overall_accuracy = (batch_true_positives + batch_true_negatives) / max(1, total_videos_processed)
                    overall_precision = batch_true_positives / max(1, batch_true_positives + batch_false_positives) if (batch_true_positives + batch_false_positives) > 0 else 0
                    overall_recall = batch_true_positives / max(1, batch_true_positives + batch_false_negatives) if (batch_true_positives + batch_false_negatives) > 0 else 0
                    overall_f1 = 2 * overall_precision * overall_recall / max(0.001, overall_precision + overall_recall) if (overall_precision + overall_recall) > 0 else 0
                    
                    # Write overall summary
                    summary_file.write("\n\nOverall Summary:\n")
                    summary_file.write("=" * 50 + "\n")
                    summary_file.write(f"Total videos: {total_videos}\n")
                    summary_file.write(f"Videos processed: {total_videos_processed}\n")
                    summary_file.write(f"Total frames processed: {total_frames}\n")
                    summary_file.write(f"Total processing time: {total_processing_time:.2f} seconds\n")
                    summary_file.write(f"Average FPS: {total_frames/total_processing_time if total_processing_time > 0 else 0:.2f}\n")
                    summary_file.write(f"Total detected fall frames: {total_detected_fall_frames}\n")
                    summary_file.write(f"Total detected non-fall frames: {total_detected_non_fall_frames}\n")
                    summary_file.write(f"Video-level metrics (each video = 1 sample):\n")
                    summary_file.write(f"Overall true positives: {batch_true_positives}\n")
                    summary_file.write(f"Overall true negatives: {batch_true_negatives}\n")
                    summary_file.write(f"Overall false positives: {batch_false_positives}\n")
                    summary_file.write(f"Overall false negatives: {batch_false_negatives}\n")
                    summary_file.write(f"Overall accuracy: {overall_accuracy*100:.2f}%\n")
                    summary_file.write(f"Overall precision: {overall_precision*100:.2f}%\n")
                    summary_file.write(f"Overall recall: {overall_recall*100:.2f}%\n")
                    summary_file.write(f"Overall F1 score: {overall_f1*100:.2f}%\n")

             # Create false detection summary
            false_detection_summary_path = os.path.join('output', 'le2i_results', f'false_detections_summary_{timestamp}.txt')
            create_false_detection_summary(false_detections_data, false_detection_summary_path)
                            
            # Print overall summary to console
            print("\n" + "="*50)
            print("BATCH PROCESSING COMPLETE")
            print("="*50)
            print(f"Videos processed: {total_videos_processed}/{total_videos}")
            print(f"Total frames processed: {total_frames}")
            print(f"Total processing time: {total_processing_time:.2f} seconds")
            print(f"Average FPS: {total_frames/total_processing_time if total_processing_time > 0 else 0:.2f}")
            print(f"Total detected fall frames: {total_detected_fall_frames}")
            print(f"Total detected non-fall frames: {total_detected_non_fall_frames}")
            print(f"Video-level Performance (each video = 1 sample):")
            print(f"Overall accuracy: {overall_accuracy*100:.2f}%")
            print(f"Overall precision: {overall_precision*100:.2f}%")
            print(f"Overall recall: {overall_recall*100:.2f}%")
            print(f"Overall F1 score: {overall_f1*100:.2f}%")
            print(f"\nDetailed summary saved to: {summary_path}")
            print(f"False detection summary saved to: {false_detection_summary_path}")
        else:
            print("Operation cancelled")
    
    else:
        print("Invalid choice. Please run again and select a valid option.")

# ===============================================================================
# TELEGRAM INTEGRATION (Optional)
# ===============================================================================

def send_telegram_alert(bot_token, chat_id, message):
    """
    Send alert message via Telegram bot
    
    Args:
        bot_token: Telegram bot API token
        chat_id: Chat ID to send message to
        message: Alert message to send
    """
    try:
        import requests
        url = f"https://api.telegram.org/bot{bot_token}/sendMessage"
        data = {
            'chat_id': chat_id,
            'text': message
        }
        response = requests.post(url, data=data)
        return response.status_code == 200
    except Exception as e:
        print(f"Failed to send Telegram alert: {str(e)}")
        return False

# ===============================================================================
# MAIN EXECUTION
# ===============================================================================

if __name__ == "__main__":
    # Run interactively
    print("Enhanced Fall Detection Using YOLOv7-W6-Pose for Real-Time Elderly Monitoring")
    print("=" * 80)
    print("Based on research paper: 'Enhanced Fall Detection Using YOLOv7-W6-Pose for Real-Time Elderly Monitoring'")
    print("Authors: Eugenia Tîrziu, Ana-Mihaela Vasilevschi, Adriana Alexandru, Eleonora Tudora")
    print("Future Internet 2024, 16, 472. https://doi.org/10.3390/fi16120472")
    print("=" * 80)
    print()
    
    try:
        run_interactive()
    except KeyboardInterrupt:
        print("\nOperation interrupted by user.")
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        import traceback
        traceback.print_exc()

# ===============================================================================
# USAGE EXAMPLES
# ===============================================================================

"""
# Example 1: Basic usage with video file
detector = FallDetector(poseweights='yolov7-w6-pose.pt', device='0')
results = run_fall_detection(
    detector=detector,
    source='path/to/video.mp4',
    device='0',
    display=True,
    save_output=True
)

# Example 2: Webcam usage
results = run_fall_detection(
    poseweights='yolov7-w6-pose.pt',
    source='0',  # Webcam ID
    device='0',
    display=True,
    save_output=True
)

# Example 3: Batch processing Le2i dataset
# Use run_interactive() and select option 5

# Example 4: Interactive single video processing
# Use run_interactive() and select option 4

# Algorithm Parameters (from research paper):
# - LENGTH_FACTOR_ALPHA (α) = 0.5: Used in height condition formula (Equation 2)
# - VELOCITY_THRESHOLD = 1.0: Threshold for fall speed detection
# - LEG_ANGLE_THRESHOLD = 45: Degrees threshold for leg angles
# - TORSO_ANGLE_THRESHOLD = 50: Degrees threshold for torso orientation  
# - ASPECT_RATIO_THRESHOLD = 0.8: Width/height ratio threshold
# - CONFIDENCE_THRESHOLD = 0.4: Minimum keypoint confidence

# Key Equations from Paper:
# Equation 1: Lfactor = sqrt((xl - xTl)² + (yl - yTl)²)
# Equation 2: yl ≤ yFl + α·Lfactor (Height condition)
# Equation 3: Hbody = |yl - yFl| (Body height)
# Equation 4: Wbody = |xl - xr| (Body width)
# Equation 5: Hbody < Wbody (Aspect ratio condition)
"""# Enhanced Fall Detection Using YOLOv7-W6-Pose for Real-Time Elderly Monitoring
# Complete Implementation Notebook

# ===============================================================================
# IMPORTS AND DEPENDENCIES
# ===============================================================================

import cv2
import time
import torch
import numpy as np
import os
import math
import re
from collections import deque
from utils.datasets import letterbox
from utils.torch_utils import select_device
from models.experimental import attempt_load
from utils.plots import output_to_keypoint, plot_skeleton_kpts
from utils.general import non_max_suppression_kpt, strip_optimizer
from torchvision import transforms

# ===============================================================================
# FALL DETECTOR CLASS
# ===============================================================================

class FallDetector:
    def __init__(self, poseweights='yolov7-w6-pose.pt', device='0'):
        """
        Initialize the Fall Detector with parameters as defined in the paper
        "Enhanced Fall Detection Using YOLOv7-W6-Pose for Real-Time Elderly Monitoring"
        
        Key parameters based on the research paper:
        - LENGTH_FACTOR_ALPHA (α): Used in height condition formula (Section 3.1)
        - VELOCITY_THRESHOLD: Threshold for fall speed detection (Section 3.2)
        - LEG_ANGLE_THRESHOLD: Degrees threshold for leg angles (Section 3.2)
        - TORSO_ANGLE_THRESHOLD: Degrees threshold for torso orientation (Section 3.2)
        - ASPECT_RATIO_THRESHOLD: Width/height ratio threshold (Section 3.1)
        - CONFIDENCE_THRESHOLD: Minimum keypoint confidence for reliable detection
        """
        print(f"Initializing Fall Detector with weights: {poseweights} on device: {device}")
        
        # Select the appropriate device
        self.device = select_device(device)
        self.half = self.device.type != 'cpu'
        
        # Load model
        self.model = attempt_load(poseweights, map_location=self.device)
        self.model.eval()
        
        # Create output directory if it doesn't exist
        os.makedirs('output', exist_ok=True)
        
        # ADJUSTED PARAMETERS based on false detection analysis
        self.LENGTH_FACTOR_ALPHA = 0.6  # Increased from 0.5 (less sensitive to height changes)
        self.VELOCITY_THRESHOLD = 0.8    # Decreased from 1.0 (catch slower falls)
        self.LEG_ANGLE_THRESHOLD = 50    # Increased from 45 (allow more leg variation)
        self.TORSO_ANGLE_THRESHOLD = 45  # Decreased from 50 (more sensitive to torso changes)
        self.ASPECT_RATIO_THRESHOLD = 0.9 # Increased from 0.8 (stricter horizontal check)
        self.CONFIDENCE_THRESHOLD = 0.4  # Keep the same
        self.MINIMUM_FALL_FRAMES = 8     # Increased from 5 (reduce false positives)

        # Add new parameters for better detection
        self.FALL_CONFIRMATION_FRAMES = 3  # Consecutive frames needed to confirm fall
        self.RECOVERY_THRESHOLD = 10       # Frames to wait before resetting fall state
        
        # State tracking variables
        self.prev_keypoints = None
        self.velocity_buffer = deque(maxlen=3)  # tracks vertical speed
        self.fall_buffer = deque(maxlen=2)      # confirmation buffer
        self.prev_frame_time = None
        self.fall_start_time = None
        self.prev_shoulder_y = None

        # Enhanced state tracking
        self.consecutive_fall_frames = 0
        self.recovery_counter = 0
        self.fall_confidence_buffer = deque(maxlen=5)  # Track confidence over time

        
        # Fall detection status
        self.fall_detected = False

    def detect_fall_enhanced(self, keypoints):
        """Enhanced fall detection with stricter criteria"""
        # ... existing keypoint extraction ...
        
        # Add additional checks:
        
        # 1. Check for sudden height drop (enhanced velocity check)
        if self.prev_keypoints is not None:
            height_drop = current_shoulder_y - self.prev_shoulder_y
            if height_drop > 20:  # Significant drop in pixels
                self.fall_confidence_buffer.append(1.0)
            else:
                self.fall_confidence_buffer.append(0.0)
        
        # 2. Check body center of mass shift
        body_center_y = (min_shoulder_y + max_feet_y) / 2
        if self.prev_body_center_y is not None:
            center_shift = body_center_y - self.prev_body_center_y
            if center_shift > 15:  # Center of mass dropped significantly
                conditions_met += 0.5
        
        # 3. Enhanced consecutive frame checking
        if is_fall:
            self.consecutive_fall_frames += 1
        else:
            self.consecutive_fall_frames = max(0, self.consecutive_fall_frames - 1)
        
        # Only confirm fall if consecutive frames show fall
        final_detection = (
            conditions_met >= 2 and 
            self.consecutive_fall_frames >= self.FALL_CONFIRMATION_FRAMES
        )
        
        # 4. Add recovery period to avoid re-triggering
        if final_detection:
            self.recovery_counter = self.RECOVERY_THRESHOLD
        elif self.recovery_counter > 0:
            self.recovery_counter -= 1
            final_detection = False  # Don't detect new falls during recovery
    
    # Environment-specific parameter sets
    ENVIRONMENT_PARAMS = {
        'Office': {
            'MINIMUM_FALL_FRAMES': 10,  # Higher threshold for office (more bending activities)
            'TORSO_ANGLE_THRESHOLD': 40,  # Stricter angle for office falls
        },
        'Coffee_room': {
            'VELOCITY_THRESHOLD': 0.7,  # Lower velocity for coffee room falls
            'LEG_ANGLE_THRESHOLD': 55,  # More relaxed for varied activities
        },
        'Home': {
            'ASPECT_RATIO_THRESHOLD': 0.85,  # Balanced for home environment
            'FALL_CONFIRMATION_FRAMES': 4,  # Slightly higher confirmation
        },
        'Lecture_room': {
            'LENGTH_FACTOR_ALPHA': 0.55,  # Balanced for lecture room
            'MINIMUM_FALL_FRAMES': 7,  # Medium threshold
        }
    }
    
    def calculate_euclidean_distance(self, point1, point2):
        """
        Calculate Euclidean distance between two points
        Used in the paper to measure distances between key body points,
        particularly for the Lfactor (length factor) calculation in Section 3.1
        
        Args:
            point1, point2: Coordinate points (x,y)
        Returns:
            Euclidean distance between the points
        """
        return math.hypot(point1[0]-point2[0], point1[1]-point2[1])

    def calculate_angle(self, a, b, c):
        """
        Calculate angle between three points (in degrees)
        Used in the paper for calculating leg angles (Section 3.2)
        
        Args:
            a, b, c: Three points where b is the vertex
        Returns:
            Angle in degrees
        """
        try:
            ba = np.array([a[0]-b[0], a[1]-b[1]])
            bc = np.array([c[0]-b[0], c[1]-b[1]])
            cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
            return np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))
        except:
            return 180  # return maximum angle if calculation fails

    def calculate_torso_angle(self, shoulders, hips):
        """
        Calculate torso angle relative to vertical axis
        Implements the torso orientation assessment described in Section 3.2
        of the paper to detect when the torso is horizontal (fallen state)
        
        Args:
            shoulders: list of shoulder points [(x,y), (x,y)]
            hips: list of hip points [(x,y), (x,y)]
        Returns:
            angle in degrees between torso and vertical axis
        """
        shoulder_center = np.mean(shoulders, axis=0)
        hip_center = np.mean(hips, axis=0)
        vertical_vector = np.array([0, 1])
        torso_vector = np.array([hip_center[0]-shoulder_center[0], 
                                hip_center[1]-shoulder_center[1]])
        
        if np.linalg.norm(torso_vector) < 1e-6:
            return 90  # neutral angle if points overlap
            
        cosine = np.dot(torso_vector, vertical_vector) / (np.linalg.norm(torso_vector) + 1e-6)
        return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

    def detect_fall(self, keypoints):
        """
        Main fall detection function implementing the paper's algorithm from Sections 3.1 and 3.2
        Combines multiple conditions (height, velocity, angles, aspect ratio) to detect falls
        
        Args:
            keypoints: Array of 17 keypoints with (x,y,confidence)
        Returns:
            tuple: (is_fall, state, condition_info)
        """
        # Keypoint indices as defined in the paper
        NOSE = 0
        LEFT_SHOULDER = 5
        RIGHT_SHOULDER = 6
        LEFT_HIP = 11
        RIGHT_HIP = 12
        LEFT_KNEE = 13
        RIGHT_KNEE = 14
        LEFT_ANKLE = 15
        RIGHT_ANKLE = 16
        
        try:
            # Extract keypoints with confidence check
            kp = {}
            
            # Reshape keypoints to get (x, y, conf) format for each keypoint
            reshaped_kpts = keypoints.reshape(-1, 3)
            
            # Extract specific keypoints
            kp['nose'] = reshaped_kpts[NOSE]
            kp['left_shoulder'] = reshaped_kpts[LEFT_SHOULDER]
            kp['right_shoulder'] = reshaped_kpts[RIGHT_SHOULDER]
            kp['left_hip'] = reshaped_kpts[LEFT_HIP]
            kp['right_hip'] = reshaped_kpts[RIGHT_HIP]
            kp['left_knee'] = reshaped_kpts[LEFT_KNEE]
            kp['right_knee'] = reshaped_kpts[RIGHT_KNEE]
            kp['left_ankle'] = reshaped_kpts[LEFT_ANKLE]
            kp['right_ankle'] = reshaped_kpts[RIGHT_ANKLE]
            
            # Confidence check for all keypoints
            if any(point[2] < self.CONFIDENCE_THRESHOLD for point in kp.values()):
                return False, "low_confidence", []

            # Get coordinates (convert to tuples for clarity)
            ls = (kp['left_shoulder'][0], kp['left_shoulder'][1])
            rs = (kp['right_shoulder'][0], kp['right_shoulder'][1])
            lh = (kp['left_hip'][0], kp['left_hip'][1])
            rh = (kp['right_hip'][0], kp['right_hip'][1])
            lk = (kp['left_knee'][0], kp['left_knee'][1])
            rk = (kp['right_knee'][0], kp['right_knee'][1])
            la = (kp['left_ankle'][0], kp['left_ankle'][1])
            ra = (kp['right_ankle'][0], kp['right_ankle'][1])

            """ 1. HEIGHT CONDITION (Paper Section 3.1) """
            # Calculate length factor (Lfactor) as Euclidean distance (Equation 1)
            torso_mid = ((lh[0] + rh[0])/2, (lh[1] + rh[1])/2)
            Lfactor = self.calculate_euclidean_distance(ls, torso_mid)
            
            # Get vertical positions
            max_feet_y = max(la[1], ra[1])
            min_shoulder_y = min(ls[1], rs[1])
            
            # Paper's height condition: yl ≤ yFl + α·Lfactor (Equation 2)
            height_cond = min_shoulder_y >= (max_feet_y - self.LENGTH_FACTOR_ALPHA * Lfactor)
            
            """ 2. VELOCITY CONDITION (Paper Section 3.2) """
            current_time = time.time()
            vertical_speed = 0
            current_min_y = min(ls[1], rs[1])
            
            if self.prev_shoulder_y is not None and self.prev_frame_time is not None:
                time_elapsed = current_time - self.prev_frame_time
                if time_elapsed > 0:
                    vertical_speed = (current_min_y - self.prev_shoulder_y) / time_elapsed
                    self.velocity_buffer.append(abs(vertical_speed))
            
            avg_speed = sum(self.velocity_buffer)/len(self.velocity_buffer) if self.velocity_buffer else 0
            speed_cond = avg_speed >= self.VELOCITY_THRESHOLD
            
            """ 3. ANGLE CONDITIONS (Paper Section 3.2) """
            left_leg_angle = self.calculate_angle(lh, lk, la)
            right_leg_angle = self.calculate_angle(rh, rk, ra)
            leg_angle_cond = min(left_leg_angle, right_leg_angle) < self.LEG_ANGLE_THRESHOLD
            
            # Torso orientation
            torso_angle = self.calculate_torso_angle([ls, rs], [lh, rh])
            torso_cond = torso_angle > self.TORSO_ANGLE_THRESHOLD
            
            """ 4. ASPECT RATIO CONDITION (Paper Section 3.1) """
            # Body orientation ratio: width/height (Equations 3, 4, 5)
            body_width = abs(ls[0] - rs[0])  # Wbody (Equation 4)
            head_to_feet = abs(kp['nose'][1] - max_feet_y)  # Hbody (Equation 3)
            orientation_ratio = body_width / (head_to_feet + 1e-6)
            aspect_cond = orientation_ratio > self.ASPECT_RATIO_THRESHOLD  # Hbody < Wbody (Equation 5)
            
            """ FALL DECISION LOGIC (Paper Section 3) """
            # Combined conditions - at least 2 must be true
            conditions_met = sum([height_cond, speed_cond, leg_angle_cond, torso_cond, aspect_cond])
            
            # State determination
            current_state = "normal"
            conditions_info = []
            
            if height_cond:
                if speed_cond:  # Rapid descent
                    current_state = "falling"
                    self.fall_start_time = current_time
                    conditions_info.append(f"speed:{avg_speed:.1f}px/s")
                elif torso_cond and self.fall_start_time and (current_time - self.fall_start_time < 1.0):
                    current_state = "fallen"
                    conditions_info.append("horizontal")
            
            if leg_angle_cond:
                conditions_info.append(f"leg_angle:{min(left_leg_angle, right_leg_angle):.0f}°")
            
            if aspect_cond:
                conditions_info.append(f"aspect:{orientation_ratio:.2f}")
            
            # Final decision with confirmation buffer
            is_fall = conditions_met >= 2
            self.fall_buffer.append(is_fall)
            final_detection = sum(self.fall_buffer) >= 2 if len(self.fall_buffer) >= 1 else is_fall
            
            if final_detection:
                current_state = "fallen"
                self.fall_detected = True
            else:
                self.fall_detected = False
            
            # Update tracking variables
            self.prev_keypoints = kp
            self.prev_shoulder_y = current_min_y
            self.prev_frame_time = current_time
            
            # Diagnostic information
            conditions_info.extend([
                f"height:{'Y' if height_cond else 'N'}",
                f"speed:{'Y' if speed_cond else 'N'}",
                f"leg_angle:{'Y' if leg_angle_cond else 'N'}",
                f"torso:{'Y' if torso_cond else 'N'}",
                f"aspect:{'Y' if aspect_cond else 'N'}",
                f"conf:{min(p[2] for p in kp.values()):.2f}"
            ])
            
            return final_detection, current_state, conditions_info
            
        except Exception as e:
            print(f"Detection error: {str(e)}")
            return False, "error", [f"Error: {str(e)}"]

    def process_frame(self, frame):
        """
        Process a single frame for fall detection
        
        Args:
            frame: Video frame to process
            
        Returns:
            frame: Processed frame with detections
            is_fall: Boolean indicating whether a fall was detected
            state: Current state (normal, falling, fallen)
            condition_info: List of conditions that triggered the detection
        """
        # Preprocess image
        orig_image = frame.copy()
        image = cv2.cvtColor(orig_image, cv2.COLOR_BGR2RGB)
        
        # Resize image while maintaining aspect ratio
        frame_height, frame_width = orig_image.shape[:2]
        image = letterbox(image, (frame_width), stride=64, auto=True)[0]
        
        # Convert to tensor
        image_ = image.copy()
        image = transforms.ToTensor()(image)
        image = torch.tensor(np.array([image.numpy()]))
        
        image = image.to(self.device)
        image = image.float()
        
        # Inference
        with torch.no_grad():
            output, _ = self.model(image)
            
        # Post-process
        output = non_max_suppression_kpt(output, 0.25, 0.65, nc=self.model.yaml['nc'], nkpt=self.model.yaml['nkpt'], kpt_label=True)
        output = output_to_keypoint(output)
        
        # Convert back to BGR for display
        img = image[0].permute(1, 2, 0) * 255
        img = img.cpu().numpy().astype(np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        
        # Initialize fall status and state for this frame
        is_fall = False
        current_state = "normal"
        condition_info = []
        
        # Process each person detected
        for idx in range(output.shape[0]):
            # Draw skeleton and keypoints
            plot_skeleton_kpts(img, output[idx, 7:].T, 3)
            
            # Calculate improved bounding box based on keypoints
            kpts = output[idx, 7:].reshape(-1, 3)
            
            # Initialize with first keypoint
            x_values = [kpt[0] for kpt in kpts if kpt[2] > 0.5]  # Only use keypoints with confidence > 0.5
            y_values = [kpt[1] for kpt in kpts if kpt[2] > 0.5]
            
            if x_values and y_values:  # Check if we have valid keypoints
                xmin, ymin = min(x_values), min(y_values)
                xmax, ymax = max(x_values), max(y_values)
                
                # Add padding to make bounding box a bit larger
                padding = 10
                xmin = max(0, xmin - padding)
                ymin = max(0, ymin - padding)
                xmax = xmax + padding
                ymax = ymax + padding
                
                # Calculate aspect ratio for reference (not used in detection)
                width = xmax - xmin
                height = ymax - ymin
                bbox_aspect_ratio = width / height if height > 0 else 0
                
                # Calculate center
                cx = int((xmin + xmax) // 2)
                cy = int((ymin + ymax) // 2)
                
                # For debugging: show aspect ratio on frame
                cv2.putText(img, f"Ratio: {bbox_aspect_ratio:.2f}", (10, 30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            else:
                # Fallback to original bounding box if no valid keypoints
                x1, y1, x2, y2 = output[idx, 0], output[idx, 1], output[idx, 2], output[idx, 3]
                xmin, ymin = x1, y1
                xmax, ymax = x2, y2
                cx, cy = int((x1 + x2) // 2), int((y1 + y2) // 2)
            
            # Get key points for this person
            key_points = output[idx, 7:]
            
            # Detect fall for this person using enhanced algorithm
            person_fall, person_state, person_conditions = self.detect_fall(key_points)
            
            # If any person is falling, set global fall status
            if person_fall:
                is_fall = True
                current_state = person_state
                condition_info = person_conditions
                
                # Add visual indication of fall - REVISED: Show "Fall Detected!" instead of metrics
                status_text = "FALL DETECTED!"
                cv2.putText(img, status_text, (50, 50), 
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                
                # Draw the bounding box in red for a fall
                cv2.rectangle(img, (int(xmin), int(ymin)), (int(xmax), int(ymax)), (0, 0, 255), 2)
                
                # Add a colored rectangle at the center
                cv2.rectangle(img, (cx-10, cy-10), (cx+10, cy+10), (84, 61, 247), -1)
                
                # Add condition info to the frame (debug information)
                for i, cond in enumerate(person_conditions[:3]):  # Show first 3 conditions only
                    cv2.putText(img, cond, (10, 60 + i*25), 
                              cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 1)
            else:
                # Draw normal bounding box in green for no fall
                cv2.rectangle(img, (int(xmin), int(ymin)), (int(xmax), int(ymax)), (0, 255, 0), 1)
                
                # Show normal state
                cv2.putText(img, f"State: {person_state}", (10, 60), 
                          cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        return img, is_fall, current_state, condition_info

# ===============================================================================
# UTILITY FUNCTIONS
# ===============================================================================

def natural_sort_key(text):
    """
    Create a sorting key that handles numbers within strings naturally.
    For example: video (1).avi, video (2).avi, ..., video (10).avi, video (11).avi
    """
    def convert(text):
        return int(text) if text.isdigit() else text.lower()
    
    return [convert(c) for c in re.split('([0-9]+)', text)]

def create_false_detection_summary(false_detections_data, summary_path):
    """
    Create a summary file containing only false positives and false negatives
    
    Args:
        false_detections_data: List of dictionaries containing video info and detection results
        summary_path: Path to save the false detection summary
    """
    with open(summary_path, 'w') as f:
        f.write(f"False Detection Summary - {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("=" * 80 + "\n\n")
        
        # Separate false positives and false negatives
        false_positives = [d for d in false_detections_data if d['type'] == 'FP']
        false_negatives = [d for d in false_detections_data if d['type'] == 'FN']
        
        # Write False Positives section
        f.write(f"FALSE POSITIVES ({len(false_positives)} videos):\n")
        f.write("-" * 50 + "\n")
        if false_positives:
            for fp in false_positives:
                f.write(f"\nVideo: {fp['video_name']} ({fp['environment']})\n")
                f.write(f"Full path: {fp['video_path']}\n")
                f.write(f"Expected: Non Fall, Detected: Fall\n")
                f.write(f"Number of frames incorrectly detected as falls: {fp['total_false_frames']}\n")
        else:
            f.write("No false positives detected.\n")
        
        # Write False Negatives section
        f.write(f"\n\nFALSE NEGATIVES ({len(false_negatives)} videos):\n")
        f.write("-" * 50 + "\n")
        if false_negatives:
            for fn in false_negatives:
                f.write(f"\nVideo: {fn['video_name']} ({fn['environment']})\n")
                f.write(f"Full path: {fn['video_path']}\n")
                f.write(f"Expected: {fn['expected_type']}, Detected: Non Fall\n")
                if fn['expected_type'] == "Fall":
                    f.write(f"This is a Fall video that was not detected\n")
                else:
                    f.write(f"This is a Non Fall video (should not have false negatives)\n")
        else:
            f.write("No false negatives detected.\n")
        
        # Write summary statistics
        f.write(f"\n\nSUMMARY STATISTICS:\n")
        f.write("=" * 50 + "\n")
        f.write(f"Total False Positives: {len(false_positives)}\n")
        f.write(f"Total False Negatives: {len(false_negatives)}\n")
        f.write(f"Total Videos with False Detections: {len(false_positives) + len(false_negatives)}\n")

def get_video_metric_note(tp, tn, fp, fn, is_fall_video=None, detected_fall_frames=None, min_fall_frames=10):
    """
    Generate a descriptive note based on video metrics
    
    Args:
        tp, tn, fp, fn: True/False Positives/Negatives
        is_fall_video: Boolean indicating if this is a Fall video (True), Non Fall video (False), or unknown (None)
        detected_fall_frames: Number of frames detected as falls
        min_fall_frames: Minimum fall frames required for fall classification
    """
    # Handle cases with detected frames below threshold
    if detected_fall_frames is not None and detected_fall_frames > 0 and detected_fall_frames < min_fall_frames:
        if is_fall_video is True and fn == 1:
            return f"Detected {detected_fall_frames} fall frames (below threshold of {min_fall_frames}) - Insufficient for fall confirmation"
        elif is_fall_video is False and tn == 1:
            return f"Detected {detected_fall_frames} fall frames (below threshold of {min_fall_frames}) - Correctly ignored as non-fall"
        else:
            return f"Detected {detected_fall_frames} fall frames (below threshold of {min_fall_frames})"
    
    # Existing logic for normal cases
    if tp == 1 and tn == 0 and fp == 0 and fn == 0:
        return f"Successfully detected fall in a Fall video ({detected_fall_frames} fall frames)"
    elif tp == 0 and tn == 1 and fp == 0 and fn == 0:
        if detected_fall_frames == 0:
            return "Successfully detected no falls in a Non Fall video"
        else:
            return f"Successfully classified as Non Fall video (only {detected_fall_frames} fall frames detected)"
    elif tp == 0 and tn == 0 and fp == 1 and fn == 0:
        return f"Incorrectly detected falls in a Non Fall video ({detected_fall_frames} fall frames) - False Positive"
    elif tp == 0 and tn == 0 and fp == 0 and fn == 1:
        if detected_fall_frames == 0:
            return "Failed to detect any falls in a Fall video - False Negative"
        else:
            return f"Detected insufficient fall frames ({detected_fall_frames}) in a Fall video - False Negative"
    else:
        return "Unexpected metric combination - please check video processing"

# ===============================================================================
# VIDEO PROCESSING FUNCTIONS
# ===============================================================================

def run_fall_detection(poseweights='yolov7-w6-pose.pt', source='pose.mp4', device='cpu', display=True, save_output=True, 
                     save_detections=True, detector=None, batch_mode=False, interactive_mode=False):
    """
    Run fall detection on a video or webcam feed
    
    Args:
        poseweights: Path to the YOLOv7 pose weights
        source: Path to video file or webcam ID (0, 1, etc.)
        device: Device to run inference on ('cpu' or '0', '1', etc. for GPU)
        display: Whether to show video with detections in real-time
        save_output: Whether to save the output video
        save_detections: Whether to save frame-by-frame detections to a file
        detector: Optional pre-initialized FallDetector instance (for batch processing)
        batch_mode: Whether this is being run as part of a batch process (suppresses some output)
    """
    # Initialize the fall detector or use the provided one
    if detector is None:
        detector = FallDetector(poseweights=poseweights, device=device)
    
    # Parse the input source
    input_path = source
    if source.isnumeric():
        input_path = int(source)
    
    # Open video capture
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        error_msg = f"Error: Could not open video source {source}"
        if not batch_mode and not interactive_mode:
            print(error_msg)
        return {
            'frames_processed': 0,
            'average_fps': 0,
            'total_fall_frames': 0,
            'total_non_fall_frames': 0,
            'detected_fall_frames': 0,
            'detected_non_fall_frames': 0,
            'true_positives': 0,
            'true_negatives': 0,
            'false_positives': 0,
            'false_negatives': 0,
            'accuracy': -1,
            'precision': -1,
            'recall': -1,
            'f1_score': -1,
            'output_path': None,
            'detections_path': None,
            'error': error_msg
        }
    
    # Get video properties and validate
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Validate video properties
    if frame_width <= 0 or frame_height <= 0 or fps <= 0:
        error_msg = f"Invalid video properties: width={frame_width}, height={frame_height}, fps={fps}"
        if not batch_mode and not interactive_mode:
            print(error_msg)
        cap.release()
        return {
            'frames_processed': 0,
            'average_fps': 0,
            'total_fall_frames': 0,
            'total_non_fall_frames': 0,
            'detected_fall_frames': 0,
            'detected_non_fall_frames': 0,
            'true_positives': 0,  # ADD THESE
            'true_negatives': 0,
            'false_positives': 0,
            'false_negatives': 0,
            'accuracy': -1,
            'precision': -1,
            'recall': -1,
            'f1_score': -1,
            'error': error_msg
        }
    
    # Debug info for problematic videos
    if not batch_mode and not interactive_mode:
        print(f"Video properties: {frame_width}x{frame_height}, {fps} fps, {total_frames} frames")
    
    # Check if we're processing a Le2i dataset video
    is_le2i = False
    expected_fall_type = None
    is_fall_video = None
    annotation_file = None
    ground_truth_fall_frames = []
    detected_fall_frames = 0
    detected_non_fall_frames = 0
    total_fall_frames = 0
    total_non_fall_frames = 0
    frame_detections = []
    
    # Check if it's a Le2i video by looking at the path
    if isinstance(input_path, str) and "Le2i_Sorted" in input_path and "Videos" in input_path:
        is_le2i = True
        
        # Create a dedicated output folder for Le2i dataset results
        le2i_output_dir = os.path.join('output', 'le2i_results')
        os.makedirs(le2i_output_dir, exist_ok=True)
        
        # Extract environment and Fall/Non Fall type from path for folder organization
        path_parts = input_path.split(os.sep)
        try:
            # Find Fall or Non Fall in the path
            fall_idx = -1
            for i, part in enumerate(path_parts):
                if part in ["Fall", "Non Fall"]:
                    fall_idx = i
                    expected_fall_type = part  # Save whether this is a Fall or Non Fall video
                    break
            
            if fall_idx >= 0 and fall_idx + 1 < len(path_parts):
                fall_type = path_parts[fall_idx]
                env_type = path_parts[fall_idx + 1]
                
                # Create subfolder for this environment and fall type
                env_fall_dir = os.path.join(le2i_output_dir, f"{env_type}_{fall_type}")
                os.makedirs(env_fall_dir, exist_ok=True)
            else:
                env_fall_dir = le2i_output_dir
        except:
            env_fall_dir = le2i_output_dir
            
        video_filename = os.path.basename(input_path)
        
        # Construct path to annotation file by replacing Videos with Annotation_files and changing extension
        annotation_path = input_path.replace("Videos", "Annotation_files").rsplit(".", 1)[0] + ".txt"
        
        # Check if annotation file exists
        if os.path.exists(annotation_path):
            try:
                with open(annotation_path, 'r') as f:
                    lines = f.readlines()
                    
                    # The first 2 lines might be metadata (number of frames, etc.)
                    # Skip them if they don't contain fall annotations
                    data_lines = []
                    for line in lines:
                        # Try to parse as comma-separated values
                        if ',' in line:
                            data_lines.append(line)
                        # Also try space-separated values
                        elif len(line.strip().split()) >= 2:
                            # Convert space-separated to comma-separated
                            values = line.strip().split()
                            data_lines.append(','.join(values))
                    
                    for line in data_lines:
                        parts = line.strip().split(',')
                        if len(parts) >= 2:
                            try:
                                # Format can be "frame_number,label,x,y,width,height" or similar
                                frame_num = int(parts[0])
                                label = int(parts[1])  # 1 for Fall, other values for no fall
                                
                                # FIXED: For Non Fall videos, annotation labels might be different
                                # In Non Fall videos, we expect NO fall frames, so any positive label indicates an issue
                                if expected_fall_type == "Non Fall":
                                    # For Non Fall videos, we don't expect any fall frames
                                    # But if annotation says there's a fall (label 1,7,8), it's likely an error
                                    # We'll treat it as no fall to be consistent with expected video type
                                    pass  # Don't add any frames to ground_truth_fall_frames for Non Fall videos
                                else:
                                    # For Fall videos, add frames with fall labels
                                    if label in [1, 7, 8]:  # Common fall labels in Le2i
                                        ground_truth_fall_frames.append(frame_num)
                            except (ValueError, IndexError):
                                # Skip lines that can't be parsed correctly
                                continue
                
                if not batch_mode:
                    if expected_fall_type == "Non Fall":
                        print(f"Processing Non Fall video - expecting no fall frames")
                    else:
                        print(f"Loaded {len(ground_truth_fall_frames)} annotated fall frames from {annotation_path}")
            except Exception as e:
                print(f"Error loading annotation file: {str(e)}")
        else:
            if not batch_mode:
                print(f"Warning: Annotation file not found at {annotation_path}")
            is_le2i = False
    
    # Setup output video writer if requested
    out = None
    if save_output:
        if isinstance(input_path, int):
            # For webcam
            output_path = os.path.join('output', f"webcam_fall_detection.mp4")
        elif is_le2i:
            # For Le2i dataset videos, save to the dedicated folder
            video_name = os.path.basename(input_path).split('.')[0]
            output_path = os.path.join(env_fall_dir, f"{video_name}_fall_detection.mp4")
        else:
            # For regular video files
            filename = os.path.basename(input_path).split('.')[0]
            output_path = os.path.join('output', f"{filename}_fall_detection.mp4")
        
        # Create VideoWriter
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))
        if not batch_mode:
            print(f"Output will be saved to: {output_path}")
    
    # Process video frames
    frame_count = 0
    total_fps = 0
    
    # For performance tracking - REVISED: Count frames not metrics
    detected_fall_frames = 0
    detected_non_fall_frames = 0
    total_fall_frames = len(ground_truth_fall_frames) if is_le2i else 0
    total_non_fall_frames = total_frames - total_fall_frames if is_le2i else total_frames
    
    # For detection tracking - REVISED: Track ALL frame detections (not just false ones)
    frame_detections = []  # Track each frame's detection result
    
    if not batch_mode:
        print(f"Starting fall detection on {os.path.basename(input_path) if isinstance(input_path, str) else 'webcam'}...")
        print(f"Total frames: {total_frames}")
        if is_le2i:
            print(f"Expected video type: {expected_fall_type}")
    
    # Process the video frames with a timeout mechanism to prevent hanging
    start_time = time.time()
    max_process_time = 300  # 5 minutes max per video
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            if frame_count == 0:
                error_msg = f"Could not read any frames from video: {source}"
                if not batch_mode and not interactive_mode:
                    print(error_msg)
                cap.release()
                if save_output and out is not None:
                    out.release()
                cv2.destroyAllWindows()
                # MAKE SURE THIS RETURN HAS ALL FIELDS
                return {
                    'frames_processed': 0,
                    'average_fps': 0,
                    'total_fall_frames': total_fall_frames if 'total_fall_frames' in locals() else 0,
                    'total_non_fall_frames': total_non_fall_frames if 'total_non_fall_frames' in locals() else 0,
                    'detected_fall_frames': 0,
                    'detected_non_fall_frames': 0,
                    'true_positives': 0,
                    'true_negatives': 0,
                    'false_positives': 0,
                    'false_negatives': 0,
                    'accuracy': -1,
                    'precision': -1,
                    'recall': -1,
                    'f1_score': -1,
                    'output_path': None,
                    'detections_path': None,
                    'error': error_msg
                }
            break
        
        frame_count += 1
        
        # Print progress periodically if not in batch mode
        if not batch_mode and frame_count % 10 == 0:
            print(f"Processing frame {frame_count}/{total_frames if total_frames > 0 else 'unknown'}")
        
        # Check if we've been processing too long
        current_time = time.time()
        if current_time - start_time > max_process_time:
            print(f"Warning: Processing time limit reached ({max_process_time}s). Stopping early.")
            break
        
        # Process frame for fall detection
        try:
            frame_start_time = time.time()
            processed_frame, is_fall, current_state, condition_info = detector.process_frame(frame)
            frame_end_time = time.time()
            
            # Calculate FPS
            processing_fps = 1 / (frame_end_time - frame_start_time)
            total_fps += processing_fps
            
            # Resize processed frame to match original dimensions for display and saving
            processed_frame_resized = cv2.resize(processed_frame, (frame_width, frame_height))
            
            # Add FPS info and frame count
            cv2.putText(processed_frame_resized, f"FPS: {processing_fps:.2f}", (frame_width - 150, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.putText(processed_frame_resized, f"Frame: {frame_count}", (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            
            # Count detections - REVISED: Count frames, not events
            if is_fall:
                detected_fall_frames += 1
            else:
                detected_non_fall_frames += 1
            
            # Determine ground truth for this frame if we have annotation data
            is_ground_truth_fall = False
            if is_le2i:
                if expected_fall_type == "Fall":
                    # For Fall videos, check if this frame is in the ground truth fall frames
                    is_ground_truth_fall = frame_count in ground_truth_fall_frames
                else:
                    # For Non Fall videos, no frames should be falls
                    is_ground_truth_fall = False
            
            # Store frame detection result for logging
            frame_detections.append({
                'frame': frame_count,
                'detected_fall': is_fall,
                'ground_truth_fall': is_ground_truth_fall,
                'state': current_state
            })
            
            # Check for detection outcomes if we have annotation data
            if is_le2i:
                # Determine outcome for display purposes
                if is_fall and is_ground_truth_fall:
                    outcome_text = "TRUE POSITIVE"
                    color = (0, 255, 0)  # Green for TP
                elif is_fall and not is_ground_truth_fall:
                    outcome_text = "FALSE POSITIVE"
                    color = (0, 0, 255)  # Red for FP
                elif not is_fall and is_ground_truth_fall:
                    outcome_text = "FALSE NEGATIVE"
                    color = (255, 0, 0)  # Blue for FN
                else:
                    outcome_text = "TRUE NEGATIVE"
                    color = (255, 255, 0)  # Cyan for TN
                
                # Add outcome label to frame
                cv2.putText(processed_frame_resized, outcome_text, (frame_width - 250, 60), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            
            # Display the frame if requested
            if display:
                cv2.imshow('Fall Detection', processed_frame_resized)
                
                # Exit on 'q' press
                key = cv2.waitKey(1) & 0xFF
                if key == ord('q'):
                    break
                elif key == ord('n') and batch_mode:
                    # In batch mode, allow 'n' to skip to the next video
                    print("Skipping to next video...")
                    break
            
            # Save frame to output video if requested
            if save_output and out is not None:
                out.write(processed_frame_resized)
                
        except Exception as e:
            print(f"Error processing frame {frame_count}: {str(e)}")
            # Continue to next frame
            continue
    
    # Release resources
    cap.release()
    if save_output and out is not None:
        out.release()
    cv2.destroyAllWindows()
    
    # Calculate metrics on video level correctly
    true_positives = 0
    true_negatives = 0
    false_positives = 0
    false_negatives = 0
    
    # Get the minimum fall frames threshold
    min_fall_frames = 5  # or detector.MINIMUM_FALL_FRAMES if hasattr(detector, 'MINIMUM_FALL_FRAMES') else 15
    
    if is_le2i:  # This is now safe because is_le2i is defined in this function
        # Check if detected fall frames meet minimum threshold
        video_has_sufficient_fall_frames = detected_fall_frames >= min_fall_frames
        
        if expected_fall_type == "Fall":
            if video_has_sufficient_fall_frames:
                true_positives = 1
            else:
                false_negatives = 1
                # Only print if not in batch mode
                if detected_fall_frames > 0 and not batch_mode and not interactive_mode:
                    print(f"\n⚠️  Detected only {detected_fall_frames} fall frames (minimum {min_fall_frames} required)")
                    print("    Not enough fall frames to confirm fall event - marking as False Negative")
        else:
            if not video_has_sufficient_fall_frames:
                true_negatives = 1
                if detected_fall_frames > 0 and not batch_mode and not interactive_mode:
                    print(f"\n✓  Detected only {detected_fall_frames} fall frames (below threshold of {min_fall_frames})")
                    print("    Correctly ignored as insufficient for fall classification")
            else:
                false_positives = 1
    
    # Print statistics if not in batch mode and not in interactive mode
    if frame_count > 0 and not batch_mode and not interactive_mode:
        avg_fps = total_fps / frame_count
        print(f"Processed {frame_count} frames")
        print(f"Average FPS: {avg_fps:.2f}")
        print(f"Detected fall frames: {detected_fall_frames}")
        print(f"Detected non-fall frames: {detected_non_fall_frames}")
        if save_output:
            print(f"Output saved to: {output_path}")
        
        # Print detection metrics
        if is_le2i:
            print("\nDetection Metrics:")
            print(f"True Positives: {true_positives}")
            print(f"True Negatives: {true_negatives}")
            print(f"False Positives: {false_positives}")
            print(f"False Negatives: {false_negatives}")
            print(f"Total videos processed: 1/1")
            
            # Calculate performance metrics
            accuracy = (true_positives + true_negatives) / (true_positives + true_negatives + false_positives + false_negatives) if (true_positives + true_negatives + false_positives + false_negatives) > 0 else 0
            precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
            recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
            f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Accuracy: {accuracy*100:.2f}%")
            print(f"Precision: {precision*100:.2f}%")
            print(f"Recall: {recall*100:.2f}%")
            print(f"F1 Score: {f1_score*100:.2f}%")

            # Add descriptive note
            metric_note = get_video_metric_note(
                            true_positives,
                            true_negatives,
                            false_positives,
                            false_negatives,
                            is_fall_video,
                            detected_fall_frames,  # Add this parameter
                            detector.MINIMUM_FALL_FRAMES  # Add this parameter
                        )
            print(f"Note: {metric_note}")
    
    # Create detections file path
    detections_path = None
    
    # Save frame-by-frame detections to file if we processed Le2i data
    if is_le2i and save_detections and len(frame_detections) > 0:
        # Create the detections.txt in the appropriate folder
        if 'env_fall_dir' in locals():
            detections_path = os.path.join(env_fall_dir, 'detections.txt')
        else:
            detections_path = os.path.join(le2i_output_dir, 'detections.txt')
        
        try:
            with open(detections_path, 'a') as f:
                f.write(f"\n--- Detections for {os.path.basename(input_path)} ---\n")
                
                # Write frame-by-frame results
                for detection in frame_detections:
                    frame_num = detection['frame']
                    detected = detection['detected_fall']
                    ground_truth = detection['ground_truth_fall']
                    
                    # Determine the outcome
                    if detected and ground_truth:
                        outcome = "True Positive"
                    elif detected and not ground_truth:
                        outcome = "False Positive"
                    elif not detected and ground_truth:
                        outcome = "False Negative"
                    else:
                        outcome = "True Negative"
                    
                    # Write the detection result
                    predicted = "Fall" if detected else "No Fall"
                    actual = "Fall" if ground_truth else "No Fall"
                    f.write(f"Frame {frame_num}: {outcome} - Predicted: {predicted}, Actual: {actual}\n")
                    
            if not batch_mode:
                print(f"Detections saved to: {detections_path}")
        except Exception as e:
            print(f"Error saving detections: {str(e)}")
    
    # Calculate average FPS
    avg_fps = total_fps / frame_count if frame_count > 0 else 0
    
    # Calculate performance metrics
    total_classifications = true_positives + true_negatives + false_positives + false_negatives
    accuracy = (true_positives + true_negatives) / total_classifications if total_classifications > 0 else -1
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else -1
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else -1
    f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else -1
    
    # Return statistics
    return {
        'frames_processed': frame_count if 'frame_count' in locals() else 0,
        'average_fps': avg_fps if 'avg_fps' in locals() else 0,
        'false_detections': len([d for d in frame_detections if (d['detected_fall'] != d['ground_truth_fall'])]) if is_le2i else 0,
        'false_positives': false_positives,
        'false_negatives': false_negatives,
        'true_positives': true_positives,
        'true_negatives': true_negatives,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score,
        'output_path': output_path if 'output_path' in locals() and save_output else None,
        'detections_path': detections_path if 'detections_path' in locals() else None,
        'detected_fall_frames': detected_fall_frames,
        'detected_non_fall_frames': detected_non_fall_frames,
        'total_fall_frames': total_fall_frames,
        'total_non_fall_frames': total_non_fall_frames
    }

Enhanced Fall Detection Using YOLOv7-W6-Pose for Real-Time Elderly Monitoring
Based on research paper: 'Enhanced Fall Detection Using YOLOv7-W6-Pose for Real-Time Elderly Monitoring'
Authors: Eugenia Tîrziu, Ana-Mihaela Vasilevschi, Adriana Alexandru, Eleonora Tudora
Future Internet 2024, 16, 472. https://doi.org/10.3390/fi16120472



Enter path to weights file [default: yolov7-w6-pose.pt]:  
Use GPU? (y/n) [default: y]:  
Enter GPU device ID [default: 0]:  



Select input source:
1: Video file
2: Webcam
3: Single video from Le2i dataset
4: Interactive single video processing from Le2i dataset
5: Process all Le2i dataset videos


Enter choice [1/2/3/4/5]:  5
Enter dataset root path [default: datasets]:  


Found Le2i dataset with sorted structure at datasets\le2i\Le2i_Sorted


...

Optimizer stripped from yolov7-w6-pose.pt, 161.1MB
Initializing Fall Detector with weights: yolov7-w6-pose.pt on device: 0


F:\PROJECTS\Maching Learning & Artificial Intelligence\pose-estimation\models\experimental.py:242: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(w, map_loc

Fusing layers... 


C:\Users\naufa\anaconda3\envs\cv_env\Lib\site-packages\torch\functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3596.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



Processing: Fall/Coffee_room_01/video (1).avi

Current Video Results:
Processed 157 frames
Average FPS: 49.46
Detected fall frames: 106
Detected non-fall frames: 51
Detection details saved to: output\le2i_results\Coffee_room_01_Fall\detections.txt

Individual Video Metrics:
True Positives: 1
True Negatives: 0
False Positives: 0
False Negatives: 0
Note: Successfully detected fall in a Fall video (106 fall frames)

Batch Cumulative Metrics:
True Positives: 1
True Negatives: 0
False Positives: 0
False Negatives: 0
Total videos processed: 1/48
Batch Accuracy: 100.00%
Batch Precision: 100.00%
Batch Recall: 100.00%
Batch F1 Score: 100.00%
✅ Successfully processed video (1).avi

Processing: Fall/Coffee_room_01/video (2).avi
❌ Error: Could not read any frames from video: datasets\le2i\Le2i_Sorted\Fall\Coffee_room_01\Videos\video (2).avi
Retry attempt 1 for video (2).avi
Waiting 2.5 seconds before retry...

Current Video Results:
Processed 306 frames
Average FPS: 60.55
Detected fall frames: 10